### Atom-level context label vocab construction 

In [1]:
import pandas as pd
from utils.build_grover_vocab import MolVocab

In [2]:
def build_atom_vocab(csv_path, vocab_save_path, smiles_col='st_smiles'):
    smi_list = pd.read_csv(csv_path)
    vocab = MolVocab(file_path=csv_file, num_workers=8, smiles_column=smiles_col)
    print(f"Atom vocab size", len(vocab))
    vocab.save_vocab(vocab_save_path)

In [ ]:
csv_path = './dataset/pretraining/guacamol_smiles.csv'
vocab_save_path = './vocab/guacamol_atom_vocab.pkl'

build_atom_vocab(csv_path, vocab_save_path)

### Fragment vocab construction using ReLMole fragmentation rule 

In [6]:
import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from collections import Counter
from utils.generate_subgraph import init_decomp


def build_fragment_vocab(csv_path, vocab_save_path, smiles_col="st_smiles"):
    smiles_values = pd.read_csv(csv_path, usecols=[smiles_col])
    counts = Counter()

    for smiles in tqdm(smiles_values[smiles_col].dropna()):
        smiles = smiles.strip()
        mol = Chem.MolFromSmiles(smiles) if smiles else None
        if mol is None:
            continue

        groups, fragments = init_decomp(mol, vocab={}, method="relmole")
        counts.update((fragment, len(atoms)) for atoms, fragment in zip(groups, fragments))

    with open(vocab_save_path, "w") as output:
        for (fragment, atom_count), frequency in counts.most_common():
            output.write(f"{fragment}\t{atom_count}\t{frequency}\n")

In [ ]:
csv_path = './dataset/pretraining/guacamol_smiles.csv'
vocab_save_path = 'guacamol_relmole_vocab.txt'
build_fragment_vocab(csv_path, vocab_save_path)